In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from PIL import Image

import mlflow
from mlflow.models import ModelConfig
from mlflow.models import infer_signature
from mlflow.models.resources import (DatabricksServingEndpoint, DatabricksVectorSearchIndex)

In [0]:
rag_chain_conf_path = "../conf/rag_chain_config.yml"
uc_model_conf_path = "../conf/uc_model_registry.yml"

model_config = ModelConfig(development_config=rag_chain_conf_path)
databricks_resources = model_config.get("databricks_resources")
retriever_config = model_config.get("retriever_config")

In [0]:
uc_config = ModelConfig(development_config=uc_model_conf_path)
uc_model_chain_config = uc_config.get("rag_chain_model")
uc_model_name = uc_model_chain_config.get("full_name")

## Log to MLflow

In [0]:
lc_mdoel_signature = infer_signature(model_input=model_config.get("input_example"), model_output=model_config.get("output_example"))

In [0]:
image_filename = "chain_image.png"
rag_chain_notebook_filename = "rag_chain"

# Log the model to MLflow
with mlflow.start_run():
    
    mlflow.log_image(mlflow.Image(os.path.join(os.getcwd(), image_filename)), image_filename)

    logged_chain_info = mlflow.langchain.log_model(
        lc_model=os.path.join(os.getcwd(), rag_chain_notebook_filename),
        model_config=rag_chain_conf_path,
        artifact_path="chain",
        input_example=model_config.get("input_example"),
        signature=lc_mdoel_signature,
        code_paths=["helpers/"],
        resources=[
            DatabricksServingEndpoint(endpoint_name=databricks_resources.get("model_name")),
            DatabricksVectorSearchIndex(index_name=retriever_config.get("index_name"))
        ],
        pip_requirements="requirements.txt",
    )

In [0]:
# Test the chain locally
chain = mlflow.langchain.load_model(logged_chain_info.model_uri)
chain.invoke(model_config.get("input_example"))

## Register to UC

In [0]:
mlflow.set_registry_uri("databricks-uc")

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_chain_info.model_uri, name=uc_model_name)

In [0]:
# Test the chain locally
chain = mlflow.langchain.load_model(f"models:/{uc_registered_model_info.name}/{uc_registered_model_info.version}")
chain.invoke(model_config.get("input_example"))